# Interactive choropleth — ICSE affiliations by year

Reads `data/raw/icse_affiliations.json`, counts author-institution rows by
country for each year, and shows a world choropleth with a **year slider**.
The colour scale is locked across years so maps are comparable.

In [23]:
import json
from pathlib import Path

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.cm import ScalarMappable
from shapely.geometry import MultiPolygon

import ipywidgets as widgets
from IPython.display import display

## 1. Config

In [24]:
INPUT_FILE = Path("../../data/raw/icse_affiliations.json")
TITLE      = "ICSE Author Affiliations by Country"

# colour scheme: near-white at 0 -> very dark red at the max (high contrast)
CMAP   = LinearSegmentedColormap.from_list(
    "white_darkred",
    ["#fff5f0", "#fcbba1", "#fc9272", "#fb6a4a", "#de2d26", "#a50f15", "#67000d"],
)
OCEAN  = "#cce5ff"   # sea / figure background
NODATA = "#eaeaea"   # countries with no rows that year

## 2. Load records & build `{year: {iso_a3: count}}`

In [25]:
with open(INPUT_FILE) as f:
    records = json.load(f)

df = pd.DataFrame(records)
df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")
df["country_code"] = df["country_code"].str.upper().str.strip()
df = df.dropna(subset=["year", "country_code"])

# world geometry uses ISO alpha-3; the data has alpha-2 -> bridge them
try:
    import pycountry
    def a2_to_a3(a2):
        try:
            return pycountry.countries.get(alpha_2=a2).alpha_3
        except Exception:
            return None
except ImportError:
    A2_TO_A3 = {
        "AR":"ARG","AU":"AUS","AT":"AUT","BE":"BEL","BR":"BRA","CA":"CAN",
        "CH":"CHE","CL":"CHL","CN":"CHN","CO":"COL","CZ":"CZE","DE":"DEU",
        "DK":"DNK","EG":"EGY","ES":"ESP","FI":"FIN","FR":"FRA","GB":"GBR",
        "GR":"GRC","HK":"HKG","HU":"HUN","IE":"IRL","IL":"ISR","IN":"IND",
        "IR":"IRN","IT":"ITA","JP":"JPN","KR":"KOR","LU":"LUX","MX":"MEX",
        "MY":"MYS","NL":"NLD","NO":"NOR","NZ":"NZL","PL":"POL","PT":"PRT",
        "RO":"ROU","RU":"RUS","SE":"SWE","SG":"SGP","TR":"TUR","TW":"TWN",
        "UA":"UKR","US":"USA","ZA":"ZAF",
    }
    a2_to_a3 = lambda a2: A2_TO_A3.get(a2)
    print("pycountry not found - using built-in alpha-2->3 lookup (pip install pycountry for full coverage).")

df["iso_a3"] = df["country_code"].apply(a2_to_a3)
unmatched = sorted(df.loc[df["iso_a3"].isna(), "country_code"].unique())
if unmatched:
    print("Could not map to alpha-3:", unmatched)
df = df.dropna(subset=["iso_a3"])

counts_by_year = df.groupby(["year", "iso_a3"]).size().reset_index(name="count")
data = {
    int(y): dict(zip(g["iso_a3"], g["count"]))
    for y, g in counts_by_year.groupby("year")
}

YEARS = sorted(data)
VMAX  = int(counts_by_year["count"].max())   # lock colour scale across years
print(f"{len(df):,} rows | years {YEARS[0]}-{YEARS[-1]} | max country-year count = {VMAX}")

11,109 rows | years 2010-2024 | max country-year count = 651


## 3. Load world geometry

`gpd.datasets` was removed in geopandas 1.x, so load Natural Earth from its source.

In [26]:
url = "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip"
world_full = gpd.read_file(url)

SOVEREIGN_TYPES = {"Sovereign country", "Country"}
world = world_full[world_full["TYPE"].isin(SOVEREIGN_TYPES)].copy()

# keep "ADMIN" as "name" for the France fix, drop it after
world = world.rename(columns={"ADM0_A3": "iso_a3", "ADMIN": "name"})[["iso_a3", "name", "geometry"]]

# fix France — remove French Guiana
france_geoms = list(world.loc[world["name"] == "France", "geometry"].values[0].geoms)
mainland = [g for g in france_geoms if g.centroid.x > -30]
world.loc[world["name"] == "France", "geometry"] = MultiPolygon(mainland)

world = world.drop(columns=["name"])

## 4. Interactive map — drag the year slider

In [27]:
%matplotlib widget

norm   = Normalize(vmin=0, vmax=VMAX)
BOUNDS = world.total_bounds          # lock the view to the full world
ALL    = "(all countries)"

# country name <-> ISO-3 for the search box (countries present in the data)
try:
    import pycountry
    def _name(iso):
        c = pycountry.countries.get(alpha_3=iso)
        return c.name if c else iso
except ImportError:
    def _name(iso):
        return iso

iso_in_data = sorted({iso for yr in data for iso in data[yr]})
ISO_TO_NAME = {iso: _name(iso) for iso in iso_in_data}
NAME_TO_ISO = {v: k for k, v in ISO_TO_NAME.items()}
country_options = [ALL] + sorted(NAME_TO_ISO)

def _hide_toolbar(f):
    for attr in ("toolbar_visible", "header_visible", "footer_visible"):
        try:
            setattr(f.canvas, attr, False)
        except Exception:
            pass

plt.ioff()                            # build the figures, then show them in a layout

# ---- main map figure (8 x 4) ----
fig = plt.figure(figsize=(8, 4))
fig.patch.set_facecolor("white")
_hide_toolbar(fig)
MAP    = [0.02, 0.04, 0.80, 0.80]
map_ax = fig.add_axes(MAP)
cax    = fig.add_axes([0.885, 0.14, 0.018, 0.60])
fig.suptitle(TITLE, fontsize=15, fontweight="bold", y=0.965)
cb = fig.colorbar(ScalarMappable(cmap=CMAP, norm=norm), cax=cax)
cb.set_label("Author-institution rows", fontsize=7)
cb.ax.tick_params(labelsize=6)

# ---- year slider as its own small figure, with a tick + label per year ----
sfig = plt.figure(figsize=(5.4, 0.55))
sfig.patch.set_facecolor("white")
_hide_toolbar(sfig)
sax = sfig.add_axes([0.03, 0.55, 0.94, 0.30])
year_slider = Slider(sax, "", YEARS[0], YEARS[-1], valinit=YEARS[0],
                     valstep=1, color="#dc143c")
year_slider.valtext.set_visible(False)
_x0, _w = sax.get_position().x0, sax.get_position().width
for _y in YEARS:
    _xf = _x0 + (_y - YEARS[0]) / (YEARS[-1] - YEARS[0]) * _w
    sfig.add_artist(plt.Line2D([_xf, _xf], [0.50, 0.55],
                               transform=sfig.transFigure, color="#888", lw=0.8))
    sfig.text(_xf, 0.42, str(_y), fontsize=4.5, ha="center", va="top", color="#555")

def render(*_):
    year = int(year_slider.val)
    map_ax.clear()
    map_ax.set_facecolor(OCEAN)
    map_ax.set_navigate(False)
    map_ax.format_coord = lambda x, y: ""
    w = world.copy()
    w["value"] = w["iso_a3"].map(data.get(year, {}))

    chosen_iso = NAME_TO_ISO.get(country_box.value.strip())
    if chosen_iso:                                  # highlight ONLY the chosen country
        w.plot(ax=map_ax, color=NODATA, edgecolor="white", linewidth=0.3)
        sel = w[w["iso_a3"] == chosen_iso]
        if not sel.empty and sel["value"].notna().any():
            sel.plot(column="value", ax=map_ax, cmap=CMAP, norm=norm,
                     edgecolor="black", linewidth=0.9)
        else:                                       # chosen country has no rows that year
            sel.plot(ax=map_ax, color="#ffffff", edgecolor="black", linewidth=0.9)
    else:                                           # full choropleth
        w[w["value"].isna()].plot(ax=map_ax, color=NODATA, edgecolor="white", linewidth=0.3)
        layer = w[w["value"].notna()]
        if not layer.empty:
            layer.plot(column="value", ax=map_ax, cmap=CMAP, norm=norm,
                       edgecolor="white", linewidth=0.3)

    map_ax.set_xlim(BOUNDS[0], BOUNDS[2])
    map_ax.set_ylim(BOUNDS[1], BOUNDS[3])
    map_ax.set_aspect("equal", adjustable="datalim")
    map_ax.set_position(MAP)
    map_ax.set_xticks([]); map_ax.set_yticks([])
    fig.canvas.draw_idle()

# ---- controls row:  country search  |  year slider ----
country_box = widgets.Combobox(value=ALL, options=country_options,
                               placeholder="type a country…", description="Country:",
                               ensure_option=False,
                               layout=widgets.Layout(width="190px"))
reset_btn = widgets.Button(description="Show all", layout=widgets.Layout(width="80px"))
reset_btn.on_click(lambda _: setattr(country_box, "value", ALL))

year_slider.on_changed(render)
country_box.observe(render, names="value")
render()

controls = widgets.HBox([country_box, reset_btn, sfig.canvas],
                        layout=widgets.Layout(justify_content="center",
                                              align_items="center",
                                              margin="10px 0 30px 0"))   # spacing below
display(widgets.VBox([fig.canvas, controls]))